## Add code commune to dataset

In [1]:
import pyreadstat
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import glob
import os

### Read raw .sas7bdat files, extract columns needed, save as parquet

In [2]:
# specify variables to keep 
var = [
  'SIREN',
  'NIC',
  'COMT', # 2004 onwards
  'DEPT', # 2004
  'REGT' # 2004
]


# store files in "1 - Data processing/Temp/DADS/DADS_Postes.parquet"

# valid for 2016 and 2017
list_files = ['post24','post27','post28','post32','post44','post52','post53','post75','post76','post84','post93','post94','post97','post99']

for file in list_files:
    df_2016,_ = pyreadstat.read_sas7bdat(f"//casd.fr/casdfs/Projets/F1CHOCS/Data/DADS_DADS Postes_2016/Régions/{file}.sas7bdat", usecols=var)
    df_2016.to_parquet(f"C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/DADS_Postes.parquet/2016/{file}.parquet")
    df_2017,_ = pyreadstat.read_sas7bdat(f"//casd.fr/casdfs/Projets/F1CHOCS/Data/DADS_DADS Postes_2017/Régions/{file}.sas7bdat", usecols=var)
    df_2017.to_parquet(f"C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/DADS_Postes.parquet/2017/{file}.parquet")

# valid for 2018 and 2019
list_files = ['post_1','post_2','post_3','post_4']

for file in list_files:
    df_2018,_ = pyreadstat.read_sas7bdat(f"//casd.fr/casdfs/Projets/F1CHOCS/Data/DADS_DADS Postes_2018/{file}.sas7bdat", usecols=var)
    df_2018.to_parquet(f"C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/DADS_Postes.parquet/2018/{file}.parquet")
    df_2019,_ = pyreadstat.read_sas7bdat(f"//casd.fr/casdfs/Projets/F1CHOCS/Data/DADS_DADS Postes_2019/{file}.sas7bdat", usecols=var)
    df_2019.to_parquet(f"C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/DADS_Postes.parquet/2019/{file}.parquet")


    

### Read files and save collapsed SIRE_codeCommune correspondances


In [ ]:
# store files in "1 - Data processing/Temp/DADS/SIRET-codeComune"
years = [2016,2019]

for year in years:
    
    input_path = os.path.join("C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/DADS_Postes.parquet/",str(year))
    output_path = os.path.join("C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/SIRET-codeComune/etablissements_"f"{year}.parquet")
    files = glob.glob(os.path.join(input_path, "*parquet"))

    df_binded = pd.concat([pd.read_parquet(f)[["SIREN","NIC","COMT", "REGT", "DEPT"]] for f in files], ignore_index = True)

    df_binded["SIREN"] = df_binded["SIREN"].astype(str).str.zfill(9)
    df_binded["NIC"] = df_binded["NIC"].astype(str).str.zfill(5)
    df_binded["SIRET"] = df_binded["SIREN"] + df_binded["NIC"]

    # keep only one unique combination of SIRET-codeCommune
    df_communes = df_binded.drop_duplicates(subset=["SIRET", "COMT"])

    #clean
    df_communes = df_communes.rename(columns={"COMT":"INSEE_COM"})
    df_communes["YEAR"] = year
    df_communes = df_communes[["YEAR", "SIREN", "NIC", "SIRET", "INSEE_COM", "DEPT", "REGT"]]
    df_communes = df_communes.sort_values(by=["SIRET","INSEE_COM"])



    df_communes.to_parquet(output_path, engine = "pyarrow", index = False)





In [2]:
# aggregate all years into a single file

input_path = os.path.join("C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/SIRET-codeComune/")
files = glob.glob(os.path.join(input_path, "*parquet"))

df_communes_all = pd.concat([pd.read_parquet(f) for f in files], ignore_index = True)
df_communes_all = df_communes_all.rename(columns={"YEAR":"ANNEE"})

df_communes_all = df_communes_all.dropna(subset=["INSEE_COM"])
df_communes_all.to_parquet("C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/DADS/SIRET-codeComune/all_etablissements.parquet", engine = "pyarrow", index = False)


In [ ]:
# Load main dataset
FICUSFARE_DADS = pd.read_parquet("C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/FICUSFARE_DADS/FICUSFARE_DADS_1.parquet")[["ANNEE", "SIRET", "SIREN"]]

# ... and merge
df_merged = FICUSFARE_DADS.merge(
   df_communes_all,
   on=["SIRET","ANNEE"],
   how="left"
)

df_merged.to_parquet("C:/Users/Public/Documents/Fontaine/Floods_Shock/1 - Data processing/Temp/FICUSFARE_DADS/FICUSFARE_DADS_1_geo_etab.parquet", engine = "pyarrow", index = False)
